# CARBAMYLATION ANALYSIS TOOL

This Jupyter notebook enables selecting and curating protein atomic structures for the analysis of their lysines. The aim, is to identify the subset of lysines that may be carbamylated. The pipeline is divided in the following steps.

* 1. [Data Gathering](#gather)
  - 1.1. [Mine UNIPROT database](#uniprot)
  - 1.2. [Mine PDB and AlphaFold databases](#pdb)
* 2. [Feature Extraction](#measure)
* 3. [Data Analysis](#analysis)
* 4. [Interactive Plots](#plot)

Further explanations about each of these steps are provided in the individual sections below.
Now, let's execute the cell below (press Shift+ENTER keys) to import all the Python packages required.

In [ ]:
import os

#data handling
import numpy as np
import pandas as pd
pd.set_option("display.max_rows", None, "display.max_columns", None)

#plotting
import plotly.express as px
import plotly.graph_objects as go
import nglview as nv
from ipywidgets import HBox, VBox
from ipywidgets import widgets

# atomic structures manipulation
import biobox as bb

#carbamylation analysis procedures
from uniprot import Uniprot
from protein import PDB
from measure import Measure
from analysis import Analysis
from COOLPLOTS import CoolPlots

Before getting started, let's define an output directory. If none defined, the default name will be "result".

In [ ]:
outdir="Demo"

## 1. Data Gathering  <a class="anchor" id="gather"></a>

### 1.1. Mine UNIPROT database  <a class="anchor" id="uniprot"></a>

Load data from UNIPROT database. The commands i.e. `from_csv_file` & `get_protein_data` can be launched multiple times, to compile a database saved in a pandas DataFrame (`Uniprot.df`).

For the demo, we only use two uniprot codes from the main dataset

In [ ]:
# see the data first
df = pd.read_csv('cyano_reviewed_first10.csv')
df['Uniprot Entry']

In [ ]:
UP = Uniprot()
UP.from_csv_file('cyano_reviewed_first10.csv')
UP.df

In [ ]:
# alternatively
UP.get_protein_data('Q03513')
UP.df

### 1.2. Mine PDB and AlphaFold databases  <a class="anchor" id="pdb"></a>

Load and curate all proteins described in a Uniprot DataFrame (`Uniprot.df`). Two optional parameters can be passed to the PDB constructor, `outdir` (output directory, "result" by default) and the largest acceptable gap in sequence (`gap`, 10 by default).

For each entry in the database:
* raw files are downloaded from the PDB in a subfolder within the output directory (defined by `PDB.raw_dir`, "conformations" by default)
* each raw file is cleaned: heteroatoms, hydrogens and non-protein, non-standard aminoacids are amended. Ions are preserved, as they may have important effect on the feature extraction phase
* different conformations (usually from NMR ensembles) are saved in different files
* if multiple aminoacid rotamers exist, multiple files are produced
* all produced files are patched, i.e. missing parts in the sequence are added using Modeller
* all files correctly patched are saved in a different subfolder in the output directory (defined by `PDB.curated_dir`, "curated" by default).

This process will generate a new DataFrame, `PDB.df`, containing information on all the structures that have been correctly curated.

<div class="alert alert-warning">
<b>Warning:</b> depending on how many proteins have to be downloaded and curated, this operation may take a long time. AlphaFold and PDB files not needing curation are processed in few seconds. Large multimers requiring curation can take up to 1 min. Multiple conformations in a single protein structure are individually curated.
</div>

In [ ]:
pdb = PDB(outdir = outdir, gap = 10)
pdb.gather_proteins(UP.df) # may take a few hours

Show the DataFrame reporting on structures successfully downloaded and curated, and save it into a CSV file. If no filename is given, file `proteins.csv` will be saved in the output directory. This CSV file can be later loaded with the method `PDB.load_state(filename)`. 

In [ ]:
pdb.df
#pdb.save_state()

### 2. Feature Extraction  <a class="anchor" id="measure"></a>

For all structures correctly curated in the previous step, calculate pKa and Solvent Accessible Surface Area (SASA) and save final result into a file. If no filename is given, the file will be called `measures.csv`.

Minor updates:
* When creating an instance of the Measure class, a log file will also be created, which will be responsible for documenting the data during the calculation. If not path is given for the log file, the log file will be called `measure_log.txt` and stored in the ourdir defined by the user.
* `M.restart_measure` restarts `measure_dataframe` if it was stopped by any reason.
* `M.recover_from_log` searches for the log filed and convert the data stored inside it to a dataframe.

<div class="alert alert-warning">
<b>Warning:</b> calculations typically take 5-30 sec. per curated protein structure (depending on its size and number of lysines).
</div>

In [ ]:
# logger can be turned off if not needed
M = Measure(pdb.df, outdir=outdir, activate_log = True, log_path="measure_log.txt")
M.measure_dataframe()
M.save_state()

In [ ]:
M.df.sort_values(['Uniprot Entry', 'Resid'])

## 3. Data Analysis  <a class="anchor" id="analysis"></a>

<div class="alert alert-warning">
<b>Warning:</b> From here onwards, the code is not yet fully implemented. See the markdown comments as a guide for what needs to be done.</div>

The Analysis class is resonsible of interpreting data stored in the DataFrame `Measures.df`. Data is saved in `Analysis.df` and  automatically aggregated in `Analysis.df_aggregate`. Aggregation involves calculating min, max, mean and standard deviation for each feature (a single lysine may features multiple measurements in the input dataframe).
Here, the ground truth (i.e. known carbamylated lysines) can also be loaded (this function is still to be implemented). Ground truth is appended as an extra column to `Analysis.df` and `Analysis.df_aggregate`.

Start by initializing `Analysis` either via a previously saved logfile, or from a DataFrame in a `Measure` instance.

In [ ]:
df = pd.read_csv('training_data_all.csv')
analysis = Analysis(df)

In [ ]:
analysis.aggregate()
analysis.df_aggregated

A method to obtain all the data comes from structures predicted by AlphaFold

In [ ]:
analysis.get_data_alphafold()

A method to make basic plots, i.e. histogram & bixplot. Note if no uniprot code and resid id is given, the plot will use the data of the whole dataframe

In [ ]:
analysis.plot_graph(plot_type = 'histogram', 
                    feature = 'pKa', 
                    uniprot_entry = False, # can also speficy a uniprot code and a resid ID
                    resid = False)

A method to get unusual lysine data using ranges of the feature defined by the user, the output will be a dataframe

In [ ]:
analysis.get_extreme_values(feature = 'pKa', lower = 1, upper = 14)

A method to remove a sub dataframe from the main dataframe (a convenient way to remove outliers or suspicious data)

In [ ]:
df_to_remove = analysis.get_extreme_values(feature = 'pKa', lower = 1, upper = 14)
analysis.remove_df(df_to_remove)

Anther method to aggregate data: for each lysine, only one measurement of pKa & sasa will be included. The criterion of selection can either be `average` or `south_east`(the point with relatively lower pKa and higher sasa, the trade-off of the two features can be controlled by parameter `weight`, the default value is 0.5).

In [ ]:
analysis.subset(analysis.df, method = 'average')
analysis.df_sub

An important method to scrape all the GO Terms for each uniprot code from the Uniprot Database, the data will then be stored in a dictionary inside Analysis class. This operation uses multi-threading to speed up.

In [ ]:
analysis.Go_Get_Data()

In [ ]:
analysis.GO_Search_Term(analysis.df, code = '0042803', name = '')

In [ ]:
analysis.GO_Search_Protein('P73832')

## 4. Interactive Plots  <a class="anchor" id="plot"></a>

Once we have scraped all the GO Terms, we pass the `analysis` object to a new class `CoolPlots`

In [ ]:
cp = CoolPlots(analysis, outdir = outdir)

In [ ]:
"""checking for enrichment within a range"""
pka_range = [1.0, 8.8]
sasa_range = [0.0, 100.0]
df = cp.enrichment_analysis(pka_range, sasa_range)
display(df)

`advanced_plot` can be used in the following ways:

* without selecting any region, one can examine a GO Term shown in the dropdown lists and then click one of the red points (these are the average or the south_east measurement of the lysines that come from the proteins that have the GO Term). Finally, click a green point (an actual measurement of a lysine based on a particular PDB file) to see the 3D structure.
* one can also first select a region using the slides of pka and sasa. Note this operation will automatically initiate enrichment analysis. A bar chart will be produced based on the results of enrichment analysis. Finally, one can carry on the same process discussed above.
* If a region is selected, the export button can save the data (including GO Terms) within the region to a csv file and store it in the outdir defined by the user
* If a region is selected, the reset button can clear the region and the results of enrichment analysis
* If a 3D visualisation is displayed, the clear button can clear it.

In [ ]:
cp.advanced_plot(export_path = 'Regional_Data.csv')